# 07 — Reading the Real Project End to End
### Everything from notebooks 1-6, applied to `pipeline.py`

You've now covered every Python concept the original SupportPilot project
uses. This notebook is the payoff: read the real files, run the real
pipeline, and trace exactly which concept from notebooks 1-6 shows up at
each step.

## 7.1 Setup

In [ ]:
import database
database.init_db(force=True)
print(f"Database seeded at {database.DB_PATH}")


## 7.2 The concept map

| Notebook | Concept | Where you'll see it below |
|---|---|---|
| 1 | Type hints, f-strings, early-return functions | Every function signature in every module |
| 2 | Dicts, lists, comprehensions, sets | `CaseReport`'s dict shape, `HARD_ESCALATE_ISSUE_TYPES` |
| 3 | Classes, `@dataclass`, Enums | `retrieval.py`'s `Chunk`/`KnowledgeRetriever`, `models.py`'s `IssueType` |
| 4 | Modules & imports | `agents.py` importing `database`, `llm_client`, `retrieval` |
| 5 | Regular expressions | `llm_client.py`'s `mock_classify` |
| 6 | `sqlite3`, `json`, `argparse`, `os.environ` | `database.py`, `pipeline.py`, `main.py` |

Let's run one ticket and connect each part of the output back to this table.

In [ ]:
from pipeline import run_ticket, pretty_print_report

report = run_ticket(
    ticket_id="T-NOTEBOOK-001",
    ticket_text="Where is my order? It's been 6 days and tracking hasn't updated.",
    customer_id="CUST001",
)
pretty_print_report(report)


## 7.3 Reading `agents.py`'s `classifier_agent`

```python
def classifier_agent(ticket_id: str, ticket_text: str) -> dict:
    return llm_client.classify_ticket(ticket_id, ticket_text)
```

- **Type hints** (notebook 1): `ticket_id: str`, `ticket_text: str`, `-> dict`
- **Module import** (notebook 4): `llm_client` was imported at the top of
  `agents.py`; this function just delegates to it
- Inside `llm_client.classify_ticket`, if `MODE == "mock"`, it calls
  `mock_classify`, which is **entirely regex** (notebook 5)

## 7.4 Reading `escalation_rules.py`'s `decide_escalation`

```python
def decide_escalation(issue_type: str, sentiment: str, urgency: str, confidence: float,
                       validation_passed: bool, order_amount: float = 0.0) -> dict:
    hard, hard_reason = check_hard_escalation(issue_type, sentiment, urgency, order_amount)
    if hard:
        return {"should_escalate": True, "reason": hard_reason, "escalation_type": "hard_category"}
    ...
```

- **Default argument** (notebook 1): `order_amount: float = 0.0`
- **Tuple unpacking** (notebook 2): `hard, hard_reason = check_hard_escalation(...)`
- **Dict literal as a return value** (notebook 2): the function returns a
  freshly built dict, not a custom class
- **Set membership check** (notebook 2): inside `check_hard_escalation`,
  `issue_type in HARD_ESCALATE_ISSUE_TYPES`

In [ ]:
# Confirm the set membership check for yourself
from escalation_rules import HARD_ESCALATE_ISSUE_TYPES
print(HARD_ESCALATE_ISSUE_TYPES)
print("fraud_suspected" in HARD_ESCALATE_ISSUE_TYPES)


## 7.5 Reading `retrieval.py`'s `KnowledgeRetriever`

```python
class KnowledgeRetriever:
    def __init__(self, kb_dir: Path = KB_DIR):
        self.chunks = _load_chunks(kb_dir)
        self._texts = [c.text for c in self.chunks]
        self._vectorizer = TfidfVectorizer(stop_words="english")
        self._matrix = self._vectorizer.fit_transform(self._texts) if self._texts else None
```

- **`Path` with a default argument** (notebooks 1 & 6): `kb_dir: Path = KB_DIR`
- **A class with real internal state** (notebook 3): `self.chunks`,
  `self._texts`, `self._vectorizer`, `self._matrix` are all built once in
  `__init__` and reused by every later `.retrieve()` call
- **List comprehension** (notebook 2): `[c.text for c in self.chunks]`
- **`Chunk`** itself, used inside `_load_chunks`, is a `@dataclass`
  (notebook 3)

In [ ]:
from retrieval import KnowledgeRetriever

kr = KnowledgeRetriever()
print(f"{len(kr.chunks)} chunks loaded")
print(type(kr.chunks[0]))     # a Chunk dataclass instance
print(kr.chunks[0])

for r in kr.retrieve("customer wants a refund for a damaged item"):
    print(f"  [{r['relevance_score']}] {r['source_doc']} — {r['section']}")


## 7.6 Proving the hard escalation rule survives rephrasing

The whole point of keeping `escalation_rules.py` as plain functions with no
model call — this is the payoff of notebook 1's early-return pattern and
notebook 2's set membership check, working together with zero randomness or
prompt-dependence.

In [ ]:
direct = run_ticket("T004", "There's a transaction on my account that I don't recognize at all.", customer_id="CUST005")
softened = run_ticket("T005", "Hey, quick one - I noticed a small charge on my account, "
                               "just wanted to double check, I'm not sure it was me.", customer_id="CUST005")

print(direct["resolution_path"], "-", direct["escalation"]["reason"])
print(softened["resolution_path"], "-", softened["escalation"]["reason"])

assert direct["resolution_path"] == "escalated"
assert softened["resolution_path"] == "escalated"
print("\nBoth escalated -- the rule held regardless of phrasing.")


## 7.7 Running the full test suite and reading its output as JSON

This exercises `json.dumps(..., default=str)` from notebook 6 for real —
`CaseReport`'s `handled_at` field is a live `datetime`.

In [ ]:
import json
from test_tickets import run_suite

run_suite()


In [ ]:
# Inspect one full report as JSON, the same way `main.py --json` does
sample = run_ticket("T-FINAL", "I'd like a refund, my shoes don't fit.", customer_id="CUST003")
print(json.dumps(sample, indent=2, default=str))


## 7.8 You've now read the whole project

Every concept from notebooks 1-6 has shown up somewhere in this walkthrough.
The exercise below is the real test of whether it stuck.

## Final exercise

Without opening any project file, write out from memory:

1. What `escalation_rules.py`'s `HARD_ESCALATE_ISSUE_TYPES` is (a set of
   what, containing what), and why a set rather than a list.
2. What `retrieval.py`'s `Chunk` is, and why it's a `@dataclass` instead of
   a plain dict.
3. One regex pattern from `llm_client.py`'s `_KEYWORD_MAP`, translated into
   plain English.
4. The full import chain from `main.py` down to `database.py` (which file
   imports which, in order).

Then open the real files and check yourself line by line.